# Polymarket BTC Inefficiency Backtest (Colab)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Repo setup (robust)
GIT_REPO_URL = ''  # e.g. 'https://github.com/<org>/<repo>.git'
REPO_DIR = '/content/BTC-Polymarket'

import os, sys
from pathlib import Path

repo_path = Path(REPO_DIR)
if GIT_REPO_URL and not repo_path.exists():
    !git clone $GIT_REPO_URL $REPO_DIR

if not repo_path.exists():
    raise FileNotFoundError(f'Repo not found at {REPO_DIR}. Set GIT_REPO_URL or update REPO_DIR.')

src_path = repo_path / 'src'
if not src_path.exists():
    raise FileNotFoundError(f'Missing src directory: {src_path}')

os.chdir(REPO_DIR)
sys.path.insert(0, str(src_path))
print('Using repo at', REPO_DIR)
print('Using src at', src_path)


In [ ]:
# Config
START_DATE = '2024-01-01'
END_DATE = '2024-04-30'
ENTRY_THRESHOLD = 0.05
EXIT_THRESHOLD = 0.01
OUT_PATH = '/content/drive/MyDrive/polymarket_btc_inefficiency/data/pm_btc_reference.parquet'
REPORT_PATH = '/content/drive/MyDrive/polymarket_btc_inefficiency/backtest_report.md'


In [ ]:
from polymarket_btc.dataset import build_dataset
from polymarket_btc.backtest import BacktestConfig, simulate_backtest, write_report

rows = build_dataset(OUT_PATH, start=START_DATE, end=END_DATE)
cfg = BacktestConfig(entry_threshold=ENTRY_THRESHOLD, exit_threshold=EXIT_THRESHOLD)
bt, metrics, bets = simulate_backtest(rows, cfg)
write_report(metrics, bets, REPORT_PATH)

print('Rows:', len(rows))
print('Dataset:', OUT_PATH)
print('Report:', REPORT_PATH)

ordered = ['scans','bets','bet_rate','mean_roi_per_bet','median_roi_per_bet','win_rate','full_loss_rate','mean_expected_edge','mean_roi_per_scan','pnl_sharpe','oos_pnl_sharpe','pnl_max_drawdown','exposure','turnover']
for k in ordered:
    if k in metrics:
        print(f'{k:22s}: {metrics[k]:.6f}')

print('\nTop 5 realized bets')
for b in sorted(bets, key=lambda x: x['roi_real'], reverse=True)[:5]:
    print('-', b['ts'], '|', b['market_title'][:60], '|', b['bet_side'], '| roi=', round(b['roi_real'],4), '| edge=', round(b['expected_edge'],4))

print('\nWorst 5 realized bets')
for b in sorted(bets, key=lambda x: x['roi_real'])[:5]:
    print('-', b['ts'], '|', b['market_title'][:60], '|', b['bet_side'], '| roi=', round(b['roi_real'],4), '| edge=', round(b['expected_edge'],4))


In [ ]:
# Minimal plotting without external dependencies
# (uses matplotlib if available in Colab runtime)
try:
    import matplotlib.pyplot as plt
    eq=[]
    x=[]
    cur=1.0
    for r in bt:
        cur *= (1+r['pnl'])
        eq.append(cur)
        x.append(r['ts'])
    plt.figure(figsize=(10,4))
    plt.plot(x, eq)
    plt.title('Equity Curve')
    plt.grid(True)
    plt.show()
except Exception as exc:
    print('Plot skipped:', exc)


In [ ]:
verdict = 'yes' if metrics.get('mean_roi_per_bet',0) > 0 and metrics.get('win_rate',0) > 0.5 else 'unclear'
survives_costs = 'yes' if metrics.get('mean_roi_per_bet', 0) > 0 else 'no/unclear'
print('=== Final Verdict ===')
print('evidence for inefficiency:', verdict)
print('whether edge survives costs:', survives_costs)
print('interpretation: prioritize bet selection quality and frequency over CAGR-like scaling metrics')
